In [ ]:
import numpy as np

from prc_fuel.clean import clean_training, mask_bad_values
from prc_fuel.evaluate import rmse
from prc_fuel.features import FEATURES, build_features, duration_s
from prc_fuel.load import load
from prc_fuel.split import split_by_time
from prc_fuel.train import train

features = [f for f in FEATURES if f not in ["avg_vertical_rate", "openap_fuel"]]

df = mask_bad_values(load("train"))
fit, val = split_by_time(df)
model = train(clean_training(fit), features)

val = val.assign(pred=model.predict(build_features(val, features)))
rmse(val["fuel_kg"], val["pred"])

In [ ]:
density = val["n_points"] / duration_s(val)
nothing = val[["alt_start", "alt_end", "avg_groundspeed"]].isna().all(axis=1)

val = val.assign(
    group=np.select([nothing, density < 0.5], ["nothing", "sparse"], "dense"),
    sq_err=(val["fuel_kg"] - val["pred"]) ** 2,
)

In [ ]:
by_group = val.groupby("group").agg(
    n=("sq_err", "size"),
    rmse=("sq_err", lambda s: s.mean() ** 0.5),
    sq_err=("sq_err", "sum"),
    fuel=("fuel_kg", "mean"),
)
by_group["share_n"] = by_group["n"] / by_group["n"].sum()
by_group["share_err"] = by_group["sq_err"] / by_group["sq_err"].sum()
by_group["rel_err"] = by_group["rmse"] / by_group["fuel"]
by_group.drop(columns="sq_err").round(3)

Reference model (#17) gives 175.8 on val_time.
Groups by trajectory: dense (>= 0.5 points/s, exploration threshold), sparse (< 0.5), nothing (no altitude at the bounds and no groundspeed).
Sparse = 33% of intervals but 69% of the squared error. Their intervals are ~3 times bigger though (751 vs 261 kg), and relative to the fuel the error is the lowest (0.34 vs 0.42 for dense). So the big error seems to come from the size of the intervals more than from the missing trajectory.
Nothing = 164 intervals (0.6%), error as big as the fuel itself (rel 1.0), but only 5% of the total. Even perfect they would take ~4 kg off the RMSE at best.
So filling trajectories isn't worth it.